In [0]:
%run ../common/environmental_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"
bronze_table1 = f"{catalog_name}.{bronze_schema}.races"
silver_table1 = f"{catalog_name}.{silver_schema}.races"

In [0]:
df_circuits = spark.read.table(bronze_table)

In [0]:
#dropping the unwanted columns
df_circuits_selected = df_circuits.drop("url")

In [0]:
circuts_renamed_df = (
    df_circuits_selected
      .withColumnsRenamed({"circuitId" : "circuit_id",
                           "circuitName" : "circuit_name",
                           "lat" : "latitude",
                           "long" : "longitude"}
))

In [0]:
df_circuits_vaild = circuts_renamed_df.filter("circuit_id is not null")

In [0]:
df_circuits_distinct = df_circuits_vaild.dropDuplicates(['circuit_id'])

In [0]:
from pyspark.sql import functions as f
df_circuits_final = (
    df_circuits_distinct
      .withColumn('circuit_name' , f.initcap(f.col('circuit_name')))
      .withColumn('locality' , f.initcap(f.col('locality')))
      .withColumn('country' , f.initcap(f.col('country')))
      .withColumn('circuit_id' , f.initcap(f.col('circuit_id')))
)

In [0]:
#wite data to silver table
(
    df_circuits_final
    .write
    .format('delta')
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
from pyspark.sql import functions as f
df_races = spark.read.table(bronze_table1)
df_races_selected = df_races.drop("url")
races_renamed_df = (
    df_races_selected
      .withColumnsRenamed({"circuitId" : "circuit_id",
                           "raceName" : "race_name",
                           "date" : "race_date"}
))
df_races_distinct = races_renamed_df.dropDuplicates(['season' , 'round'])
df_races_final = (
    df_races_distinct
      .withColumn('circuit_id' , f.initcap(f.col('circuit_id')))
      .withColumn('race_name' , f.initcap(f.col('race_name')))
)
#wite data to silver table
(
    df_races_final
    .write
    .format('delta')
    .mode("overwrite")
    .saveAsTable(silver_table1)
)